# Homework: Lists | Order Data Analysis

## Before you begin
1. In Colab, choose File > Save a copy in Drive.
2. Run the checking setup cell, then work from top to bottom.
3. Replace each `...` with your code or answer. Keep the supplied variable and function names.
4. Run each `grader.check(...)` cell for feedback. A passing check covers the supplied cases; also check your reasoning and the requested programming approach.
5. Save your work regularly. At the end, run the final total and receipt cell, download the .ipynb file, and upload it to Moodle.

All retail records in this homework are fictional teaching data, not records from any other company.


In [1]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'cases': [{'code': '>>> a = [1250, 799]\n'
                           '>>> record_order(a, 2500) is None and a == [1250, 799, 2500]\n'
                           'True',
                   'points': 1},
                  {'code': '>>> a = []\n>>> record_order(a, 0) is None and a == [0]\nTrue',
                   'points': 1}],
        'name': 'q1',
        'points': 2},
 'q2': {'cases': [{'code': '>>> a = ["KIT", "PAINT", "KIT", "GLUE"]\n'
                           '>>> unique_products(a) == ["KIT", "PAINT", "GLUE"] and a == '
                           '["KIT", "PAINT", "KIT", "GLUE"]\n'
                           'True',
                   'points': 1},
                  {'code': '>>> unique_products([])\n[]', 'points': 1}],
        'name': 'q2',
        'points': 2},
 'q3': {'cases': [{'code': '>>> order_stats([1250, 799, 2500])\n[4549, 799, 2500]',
                   'points': 1},
                  {'code': '>>> order_stats([])\n[0, 0, 0]', 'points': 1},
                  {'code': '>>> order_stats([600])\n[600, 600, 600]', 'points': 1}],
        'name': 'q3',
        'points': 3},
 'q4': {'cases': [{'code': '>>> a = [900, 100, 500]\n'
                           '>>> median_order(a) == 500 and a == [900, 100, 500]\n'
                           'True',
                   'points': 1},
                  {'code': '>>> median_order([100, 101])\n100.5', 'points': 1},
                  {'code': '>>> median_order([]) is None and median_order([200, 400]) == 300\n'
                           'True',
                   'points': 1},
                  {'code': '>>> median_order([799])\n799', 'points': 1}],
        'name': 'q4',
        'points': 4},
 'q5': {'cases': [{'code': '>>> q5_length\n3', 'points': 1},
                  {'code': '>>> a, b = [100], [200, 300]\n'
                           '>>> c = combine_orders(a, b)\n'
                           '>>> c == [100, 200, 300] and a == [100] and b == [200, 300] and c '
                           'is not a and c is not b\n'
                           'True',
                   'points': 1},
                  {'code': '>>> combine_orders([], [])\n[]', 'points': 1}],
        'name': 'q5',
        'points': 3}}

grader = Grader('10-Lists', _TESTS)
check = grader.check

print("Ready. 10-Lists loaded: 5 question(s), 14 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

Ready. 10-Lists loaded: 5 question(s), 14 point(s).
Check a question any time with, for example:  grader.check("q1")


## Question 1: Record a new order

Order amounts are stored as integer cents to avoid rounding issues. Write `record_order(orders, cents)` to append an amount to the supplied list in place and return `None`. Do not create a replacement list.

Points: 2

In [ ]:
def record_order(orders, cents):
    ...

In [ ]:
grader.check("q1")

## Question 2: Find products that sold

Write `unique_products(codes)` to return each product code once, in its first-seen order. Use a list, a loop, and membership checking. Do not use a set or dictionary. Leave the input unchanged. Example: `["KIT", "PAINT", "KIT"]` becomes `["KIT", "PAINT"]`.

Points: 2

In [ ]:
def unique_products(codes):
    ...

In [ ]:
grader.check("q2")

## Question 3: Summarise order values

Write `order_stats(cents)` to return a list `[total, smallest, largest]`. Use the built-in list aggregation functions. For no orders, return `[0, 0, 0]`. All amounts are nonnegative integer cents.

Points: 3

In [ ]:
def order_stats(cents):
    ...

In [ ]:
grader.check("q3")

## Question 4: Estimate a typical order

A few large purchases can distort an average. Write `median_order(cents)` to return the middle sorted amount for an odd number of orders, or the average of the two middle amounts for an even number. Return `None` when there are no orders. Do not change the original list. Half-cent medians are allowed because this is a statistic, not a charged price. Example: `[900, 100, 500]` gives 500; `[100, 900]` gives 500.0.

Points: 4

In [ ]:
def median_order(cents):
    ...

In [ ]:
grader.check("q4")

## Question 5: Protect the original records

Predict the output: `a = [100, 200]; b = a; b.append(300); print(len(a))`. Store it in `q5_length`. Then write `combine_orders(morning, afternoon)` to return a new list containing both batches in that order, leaving both originals unchanged.

Points: 3

In [ ]:
q5_length = ...
def combine_orders(morning, afternoon):
    ...

In [ ]:
grader.check("q5")

## Beyond the homework
Optional, not graded: describe one additional edge case and explain what the correct result should be. Test it in a separate scratch cell without changing the supplied data above.

# Submission
- Run all cells in order, including the final total and receipt cell below.
- Save your completed notebook.
- Choose File > Download > Download .ipynb and upload that file to the appropriate Moodle homework submission.
- Submit the notebook, not a screenshot or receipt alone. Your instructor re-runs your code to determine your mark.


In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
